# Fase 2 · Transformación de datos en los datasets

## Objetivo

El objetivo de esta fase consiste en ejecutar los cambios y ajustes detectados durante el Análisis Exploratorio de Datos (EDA), para unificar, limpiar y transformar los datasets seleccionados en este proyecto.

En esta etapa se trabaja principalmente en:

- abordar los duplicados,
- combinar varios datasets,
- gestionar los valores nulos,
- homogenizar categorías para asegurar la integridad semántica,
- y exportar los archivos finales ya depurados.

Este proceso permitirá tener un conjunto de datos más consistentes, comparables y listos para la siguiente fase de análisis avanzado y visualización.

In [1]:
# Importación de librerías
import pandas as pd
import numpy as np
import re

# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación de módulos de transformación 
from src.etl.load_data import load_friends_data_raw, load_friends_data_translated
from src.etl import transform as trans
from transformers import pipeline
from src.etl import column_standardizer as stan
from src.etl import dataset_sanitizer as pr



 
# Configuración para visualizar todas las columnas del DataFrame
pd.set_option('display.max_columns', None) 

In [2]:
dfs = load_friends_data_raw()

[2026-06-02 10:07:15] INFO - Cargando datasets desde: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_raw
[2026-06-02 10:07:15] INFO - → Cargando weddings_divorces_ross.csv...
[2026-06-02 10:07:15] INFO - → Cargando friends_cameos.csv...
[2026-06-02 10:07:15] INFO - → Cargando friends_emotions.csv...
[2026-06-02 10:07:15] INFO - → Cargando friends_episodes.csv...
[2026-06-02 10:07:15] INFO - → Cargando friends_sets.csv...
[2026-06-02 10:07:15] INFO - → Cargando friends_info.csv...
[2026-06-02 10:07:15] INFO - → Cargando friends_quotes.csv...
[2026-06-02 10:07:16] INFO - → Cargando friends.csv...
[2026-06-02 10:07:16] INFO - → Cargando phoebe_buffay_songs.csv...
[2026-06-02 10:07:16] INFO - → Cargando duck_and_chicken.csv...
[2026-06-02 10:07:16] INFO - Todos los datasets fueron cargados correctamente.


## 1. Transformación de  las variables numéricas (friends_quotes) de orden de float a entero (int) para mejorar la estructura. 

In [3]:
df_quotes = dfs["quotes"]

df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1.0,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0.0,1.0
1,Joey,1.0,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1.0,1.0
2,Chandler,1.0,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2.0,1.0
3,Phoebe,1.0,Monica Gets A Roommate,"Wait, does he eat chalk?",3.0,1.0
4,Phoebe,1.0,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4.0,1.0


In [4]:
df_quotes["quote_order"] = df_quotes["quote_order"].astype(int)
df_quotes["season"] = df_quotes["season"].astype(int)
df_quotes["episode_number"] = df_quotes["episode_number"].astype(int)


In [5]:
df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0,1
1,Joey,1,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1,1
2,Chandler,1,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2,1
3,Phoebe,1,Monica Gets A Roommate,"Wait, does he eat chalk?",3,1
4,Phoebe,1,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4,1


In [6]:
df_quotes.to_csv("../data_processed/friends_quotes.csv", index=False, encoding="utf-8")

## 2. Limpiar y estandarizar la columna written_by (friends_info)

In [7]:
df_info= dfs["info"]

df_info.head()

,season,episode,title,directed_by,written_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,David Crane & Marta Kauffman,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,David Crane & Marta Kauffman,1994-09-29,20.2,8.1
2,1,3,The One with the Thumb,James Burrows,Jeffrey Astrof & Mike Sikowitz,1994-10-06,19.5,8.2
3,1,4,The One with George Stephanopoulos,James Burrows,Alexa Junge,1994-10-13,19.7,8.1
4,1,5,The One with the East German Laundry Detergent,Pamela Fryman,Jeff Greenstein & Jeff Strauss,1994-10-20,18.6,8.5


In [8]:
trans.process_friends_writers(df_info)

¡Fichero corregido con éxito! Guardado en: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_processed\writers.csv (303 filas).


,season,episode,writer
0,1,1,David Crane
1,1,1,Marta Kauffman
2,1,2,David Crane
3,1,2,Marta Kauffman
4,1,3,Jeffrey Astrof
...,...,...,...
298,10,16,Ted Cohen
299,10,17,Marta Kauffman
300,10,17,David Crane
301,10,18,Marta Kauffman


In [9]:
df_info.drop("written_by", axis=1, inplace=True)


df_info.head(2)

,season,episode,title,directed_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,1994-09-29,20.2,8.1


In [10]:
df_info.to_csv("../data_processed/friends_info.csv", index=False, encoding="utf-8") 

#### Traducir las columnas

In [11]:
df_dac = dfs["dac"]

In [12]:
df_dac = stan.standardize_columns(df_dac)

In [13]:
df_dac.head()

,season,episode_number,animal,accion
0,3,3x21,Pollito,Joey lo compra
1,3,3x22,Pollito,Joey y Chandler cuidan de él.
2,3,3x22,Pato,Chandler lo rescata para que el pollito tenga ...
3,3,3x25,Pollito,Aparecen en el apartamento de los chicos.
4,3,3x25,Pato,Aparecen en el apartamento de los chicos.


In [14]:
df_dac.to_csv("../data_processed/duck_and_chicken.csv", index=False, encoding="utf-8")

In [15]:
df_cameos = dfs["cameos"]
df_cameos.head(1)

,Actor/Actriz,Personaje,Descripción/Temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel (T8)


In [16]:
# 1. Extraemos la descripción y el número de la temporada usando Regex
# El patrón busca "T" seguido de uno o más números d+ dentro de un paréntesis
df_extracted = df_cameos["Descripción/Temporada"].str.extract(r"(?P<descripcion>.*?)\s*\(T(?P<temporada>\d+)\)")

# 2. Asignamos los resultados de vuelta a nuestro DataFrame original
df_cameos["descripcion"] = df_extracted["descripcion"]
df_cameos["temporada"] = df_extracted["temporada"]

# 3. Borramos la columna vieja que ya no necesitamos
df_cameos = df_cameos.drop(columns=["Descripción/Temporada"])

# Ver el resultado
df_cameos.head(1)

,Actor/Actriz,Personaje,descripcion,temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8


In [17]:
df_cameos = stan.standardize_columns(df_cameos)

In [18]:
df_cameos.head()

,actor,author,description,season
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8
1,Bruce Willis,Paul Stevens,Padre de Elizabeth y novio de Rachel,6
2,Julia Roberts,Susie Moss,Compañera de primaria de Chandler,2
3,Charlie Sheen,Ryan,Marinero novio de Phoebe que tiene varicela,2
4,Danny DeVito,Roy,El stripper sensible en la despedida de Phoebe,10


In [19]:
df_cameos.to_csv("../data_processed/friends_cameos.csv", index=False, encoding="utf-8")

In [20]:
df_sets = dfs["sets"]

In [21]:
df_sets = stan.standardize_columns(df_sets)

### Cargamos los datasets finales

In [28]:
dfs_finales = load_friends_data_translated()

[2026-06-02 10:16:40] INFO - Cargando datasets desde: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_translated
[2026-06-02 10:16:40] INFO - → Cargando friends_weddings_divorce_ross.csv...
[2026-06-02 10:16:40] INFO - → Cargando friends_cameos.csv...
[2026-06-02 10:16:40] INFO - → Cargando friends_emotions.csv...
[2026-06-02 10:16:40] INFO - → Cargando friends_episodes.csv...
[2026-06-02 10:16:40] INFO - → Cargando friends_sets.csv...
[2026-06-02 10:16:40] INFO - → Cargando friends_info.csv...
[2026-06-02 10:16:40] INFO - → Cargando friends_quotes.csv...
[2026-06-02 10:16:41] INFO - → Cargando friends.csv...
[2026-06-02 10:16:41] INFO - → Cargando friends_songs.csv...
[2026-06-02 10:16:41] INFO - → Cargando duck_and_chicken.csv...
[2026-06-02 10:16:41] INFO - → Cargando writers.csv...
[2026-06-02 10:16:41] INFO - Todos los datasets fueron cargados correctamente.


In [29]:
df_quotesf = dfs_finales["quotes"]

In [ ]:
df_quotesf["personaje"] = df_quotesf["personaje"].str.title()

In [31]:
df_quotesf.sample(20)

,personaje,numero_episodio,titulo_episodio,cita,orden_cita,temporada
32169,Monica,10,La rutina,"Oh, lo siento cariño, ella no siente lo mismo.",18,6.0
55847,Monica,4,El pastel,Tengo un plan. Tengo un plan. Voy a clavar est...,66,10.0
45289,Joey,9,El rumor,"(entrando, usando los pantalones premamá de an...",260,8.0
26073,Joey,10,La hermana inapropiada,¡Ey!,70,5.0
2476,Chandler,11,Sra. Bing,"Shhh, ocupado sonriendo de orgullo.",52,1.0
21067,Phoebe,13,El enamoramiento de Rachel,"Sí, no lo necesito.",138,4.0
26604,Rachel,12,La risa del trabajo de Chandler,"Bueno, no lo sé porque me asusté tanto que col...",154,5.0
34491,Monica,19,El refrigerador de Joey,¿Qué es la caridad?,20,6.0
45607,Ross,11,El paso adelante de Ross,Bueno. (No estoy contento con eso.),31,8.0
21288,Monica,14,El día sucio de Joey,"(mirando por la mirilla) Ohh, ella está mirand...",94,4.0


#### Utilizar una funcion para detectar las lineas que no son diálogo y eliminarlas, creando un archivo nuevo y limpio.

In [32]:
pr.export_data_anomalies("../data_translated/friends_quotes.csv", "friends_quotes_errores.csv", ["personaje", "cita"] )

[INFO] No data anomalies were found in the specified target columns.


""


In [33]:
pr.sanitize_dataset_by_index("../data_translated/friends_quotes.csv", "friends_quotes_errores.csv", "friends_quotes_clean.csv", chunksize=50000)

FileNotFoundError: [ERROR] Anomalies tracking file not found at: friends_quotes_errores.csv

### Mapear archivo quotes con los nombres 

MNCA
FBOB
PHOE
JOEY
ROSS
CHAN


In [ ]:
dicc_nombres = {
    "Phoe": "Phoebe",
    "Mnca": "Monica",
    "Rach": "Rachel",
    "Chan": "Chandler",
    "Estl": "Estelle",
    "Waiter" : "Camarera"
}

In [44]:
df_quotesf["personaje"] = df_quotesf["personaje"].apply(lambda x: dicc_nombres.get(x, x))

In [45]:
df_quotesf.tail()

,personaje,numero_episodio,titulo_episodio,cita,orden_cita,temporada
60211,Chandler,17,"El último, partes I y II","Oh, todo estará bien.",581,10.0
60212,Rachel,17,"El último, partes I y II",(llorando) ¿Tienen que ir a la nueva casa de i...,582,10.0
60213,Monica,17,"El último, partes I y II",Tenemos algo de tiempo.,583,10.0
60214,Rachel,17,"El último, partes I y II","Bien, ¿deberíamos tomar un poco de café?",584,10.0
60215,Chandler,17,"El último, partes I y II",Seguro. ¿Dónde?,585,10.0


In [46]:
df_quotesf.to_csv("../data_translated/friends_quotes.csv", index=False, encoding="utf-8")

### normalizar la fecha del fichero info aaaa-mm-dd ponerlo como dd-mm-aaaa